# Little helper

In [1]:
def to_um(value_str):
    units = {
        'nm': 1e-3,   # nanometers to micrometers
        'um': 1,      # micrometers to micrometers
        'mm': 1e3,    # millimeters to micrometers
        'cm': 1e4,    # centimeters to micrometers
        'm':  1e6     # meters to micrometers
    }
    
    # Clean and split the input
    parts = value_str.strip().lower().split()
    if len(parts) != 2:
        raise ValueError("Input must be in the form '<number> <unit>'")

    number, unit = parts
    if unit not in units:
        raise ValueError(f"Unsupported unit: {unit}")
    
    return float(number) * units[unit]


# General import

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs

%metal_heading Genshin Impact!

# Layout

In [4]:
from qiskit_metal.qlibrary.qubits.transmon_pocket_6 import TransmonPocket6
from qiskit_metal.qlibrary.qubits.transmon_cross import TransmonCross
from qiskit_metal.qlibrary.qubits.transmon_cross_fl import TransmonCrossFL

from qiskit_metal.qlibrary.couplers.tunable_coupler_01 import TunableCoupler01

from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight

from qiskit_metal.qlibrary.lumped.cap_n_interdigital import CapNInterdigital
from qiskit_metal.qlibrary.couplers.cap_n_interdigital_tee import CapNInterdigitalTee
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee

from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.launchpad_wb_coupled import LaunchpadWirebondCoupled
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround

from qiskit_metal.qlibrary.qubits.JJ_Manhattan import jj_manhattan

In [5]:
design = metal.designs.DesignPlanar()

gui = metal.MetalGUI(design)

In [6]:
design.overwrite_enabled = True
design.chips.main

{'material': 'silicon',
 'layer_start': '0',
 'layer_end': '2048',
 'size': {'center_x': '0.0mm',
  'center_y': '0.0mm',
  'center_z': '0.0mm',
  'size_x': '9mm',
  'size_y': '6mm',
  'size_z': '-750um',
  'sample_holder_top': '890um',
  'sample_holder_bottom': '1650um'}}

In [7]:
design._chips['main']['size']['size_x'] = '5mm'
design._chips['main']['size']['size_y'] = '5mm'
design._chips['main']['size']['center_x'] = '2.5mm'
design._chips['main']['size']['center_y'] = '2.5mm'
design.variables['cpw_width'] = '10 um'
design.variables['cpw_gap'] = '6 um'

# SQuADDS param

In [8]:
from squadds.core.utils import set_huggingface_api_key

set_huggingface_api_key()

API key already exists in .env file.


In [9]:
from datasets import get_dataset_config_names
from datasets import load_dataset

configs = get_dataset_config_names("SQuADDS/SQuADDS_DB")
qubit_data = load_dataset("SQuADDS/SQuADDS_DB", configs[0])

In [10]:
components = []
component_names = []
data_types = []

for config in configs:
    try:
        components.append(config.split("-")[0])
        component_names.append(config.split("-")[1])
        data_types.append(config.split("-")[2])
    except:
        pass
    
print(components)
print(component_names)
print(data_types)

['qubit', 'cavity_claw', 'coupler', 'coupler', 'measured_device_database']
['TransmonCross', 'RouteMeander', 'NCap', 'CapNInterdigitalTee']
['cap_matrix', 'eigenmode', 'cap_matrix', 'cap_matrix']


In [11]:
from squadds import SQuADDS_DB

db = SQuADDS_DB()
db.select_system("qubit")
db.select_qubit("TransmonCross")
df = db.create_system_df()

### find Qubit_1 frequency

In [12]:
from squadds import Analyzer
analyzer = Analyzer(db)
analyzer.target_param_keys()

['qubit_frequency_GHz', 'anharmonicity_MHz']

In [13]:
analyzer = Analyzer(db)
analyzer.target_param_keys()
target_params={"qubit_frequency_GHz": 4.0, "anharmonicity_MHz": -200}
results_Qubit_1 = analyzer.find_closest(target_params=target_params,
                                       num_top=3,
                                       metric="Euclidean",
                                       display=True)
print("qubit 1 frequency found:",results_Qubit_1.values[0][-2])

qubit 1 frequency found: 4.013771480977731


In [14]:
results_Qubit_1.values[0]

array(['Eli Levenson-Falk, PhD', '2023-09-20-142547', 'LFL', 'USC',
       'Andre Kuo',
       {'aedt_hfss_capacitance': 0, 'aedt_hfss_inductance': 9.686e-09, 'aedt_q3d_capacitance': 0, 'aedt_q3d_inductance': 1e-08, 'chip': 'main', 'connection_pads': {'readout': {'claw_cpw_length': '40um', 'claw_cpw_width': '10um', 'claw_gap': '5.1um', 'claw_length': '190um', 'claw_width': '15um', 'connector_location': '90', 'connector_type': '0', 'ground_spacing': '10um'}}, 'cross_gap': '30um', 'cross_length': '210um', 'cross_width': '30um', 'gds_cell_name': 'my_other_junction', 'hfss_capacitance': 0, 'hfss_inductance': 9.686e-09, 'hfss_mesh_kw_jj': 7e-06, 'hfss_resistance': 0, 'layer': '1', 'orientation': '-90', 'pos_x': '-1500um', 'pos_y': '1200um', 'q3d_capacitance': 0, 'q3d_inductance': '10nH', 'q3d_mesh_kw_jj': 7e-06, 'q3d_resistance': 0},
       'qiskit-metal',
       {'Cj': 0, 'Lj': '10nH', '_Rj': 0, 'design_name': None, 'max_mesh_length_jj': '7um', 'max_mesh_length_port': '7um', 'plot_ansys_fi

### find Qubit_2 frequency

In [15]:
analyzer = Analyzer(db)
analyzer.target_param_keys()
target_params={"qubit_frequency_GHz": 3.8, "anharmonicity_MHz": -200}
results_Qubit_2 = analyzer.find_closest(target_params=target_params,
                                       num_top=3,
                                       metric="Euclidean",
                                       display=True)
print("qubit 2 frequency found:",results_Qubit_2.values[0][-2])

qubit 2 frequency found: 3.8241520856146796


### find Qubit_3 frequency

In [16]:
analyzer = Analyzer(db)
analyzer.target_param_keys()
target_params={"qubit_frequency_GHz": 3.9, "anharmonicity_MHz": -200}
results_Qubit_3 = analyzer.find_closest(target_params=target_params,
                                       num_top=3,
                                       metric="Euclidean",
                                       display=True)
print("qubit 3 frequency found:",results_Qubit_3.values[0][-2])

qubit 3 frequency found: 3.91895098416183


### find Qubit_4 frequency

In [17]:
analyzer = Analyzer(db)
analyzer.target_param_keys()
target_params={"qubit_frequency_GHz": 4.0, "anharmonicity_MHz": -200}
results_Qubit_4 = analyzer.find_closest(target_params=target_params,
                                       num_top=3,
                                       metric="Euclidean",
                                       display=True)
print("qubit 4 frequency found:",results_Qubit_4.values[0][-2])

qubit 4 frequency found: 4.013771480977731


### find Qubit_5 frequency

In [18]:
analyzer = Analyzer(db)
analyzer.target_param_keys()
target_params={"qubit_frequency_GHz": 4.1, "anharmonicity_MHz": -200}
results_Qubit_5 = analyzer.find_closest(target_params=target_params,
                                       num_top=3,
                                       metric="Euclidean",
                                       display=True)
print("qubit 5 frequency found:",results_Qubit_5.values[0][-2])

qubit 5 frequency found: 4.1085956448307694


### find Qubit_6 frequency

In [19]:
analyzer = Analyzer(db)
analyzer.target_param_keys()
target_params={"qubit_frequency_GHz": 4.2, "anharmonicity_MHz": -200}
results_Qubit_6 = analyzer.find_closest(target_params=target_params,
                                       num_top=3,
                                       metric="Euclidean",
                                       display=True)
print("qubit 6 frequency found:",results_Qubit_6.values[0][-2])

qubit 6 frequency found: 4.203425473399873


# The Qubits

In [20]:
TransmonCross.get_template_options(design)

{'pos_x': '0.0um',
 'pos_y': '0.0um',
 'orientation': '0.0',
 'chip': 'main',
 'layer': '1',
 'connection_pads': {},
 '_default_connection_pads': {'connector_type': '0',
  'claw_length': '30um',
  'ground_spacing': '5um',
  'claw_width': '10um',
  'claw_gap': '6um',
  'connector_location': '0'},
 'cross_width': '20um',
 'cross_length': '200um',
 'cross_gap': '20um',
 'hfss_inductance': '10nH',
 'hfss_capacitance': 0,
 'hfss_resistance': 0,
 'hfss_mesh_kw_jj': 7e-06,
 'q3d_inductance': '10nH',
 'q3d_capacitance': 0,
 'q3d_resistance': 0,
 'q3d_mesh_kw_jj': 7e-06,
 'gds_cell_name': 'my_other_junction'}

In [21]:
jj_manhattan.get_template_options(design)

{'pos_x': '0.0um',
 'pos_y': '0.0um',
 'orientation': '0.0',
 'chip': 'main',
 'layer': '1',
 'JJ_pad_lower_width': '25um',
 'JJ_pad_lower_height': '10um',
 'JJ_pad_lower_pos_x': '0',
 'JJ_pad_lower_pos_y': '0',
 'finger_lower_width': '1um',
 'finger_lower_height': '20um',
 'extension': '1um'}

### Qubit 1

In [22]:
xmon_options = dict(
    pos_x = '1.0mm',
    pos_y = '4.00mm',
    orientation = '270',
    cross_width = '24um',
    cross_length = '160um',
    cross_gap = '24um',
    connection_pads=dict(
        readout = dict(connector_location = '90', connector_type = '0', claw_length='90um')
    ),
)

transmon_options_1=results_Qubit_1.values[0][5]
transmon_options_1.update(
    pos_x = '1.0mm',
    pos_y = '4.00mm',)
Qubit_1 = TransmonCross(design, 'Qubit_1', options=xmon_options)

gui.rebuild()
gui.autoscale()
# gui.zoom_on_components(['Qubit_1'])

### Qubit 2

In [47]:
xmon_options = dict(
    pos_x = '4.0mm',
    pos_y = '3.55mm',
    orientation = '90',
    cross_width = '24um',
    cross_length = '160um',
    cross_gap = '24um',
    connection_pads=dict(
        connector_1 = dict(connector_location = '90', connector_type = '0', claw_length='90um')
    ),
)

transmon_options_2=results_Qubit_1.values[0][5]
transmon_options_2.update(
    pos_x = '4.0mm',
    pos_y = '3.55mm',
    orientation = '90',
    )
Qubit_2 = TransmonCross(design, 'Qubit_2', options=transmon_options_2)


gui.rebuild()
gui.autoscale()
gui.zoom_on_components(['Qubit_2'])

### Qubit 3

In [24]:
xmon_options = dict(
    pos_x = '1.0mm',
    pos_y = '2.85mm',
    orientation = '270',
    cross_width = '24um',
    cross_length = '160um',
    cross_gap = '24um',
    connection_pads=dict(
        connector_1 = dict(connector_location = '90', connector_type = '0', claw_length='90um')
    ),
)

transmon_options_3=results_Qubit_1.values[0][15]
transmon_options_3.update(
    pos_x = '1.0mm',
    pos_y = '2.85mm',
    orientation = '270',)

Qubit_3 = TransmonCross(design, 'Qubit_3', options=transmon_options_3)
gui.rebuild()
gui.autoscale()
gui.zoom_on_components(['Qubit_3'])

### Qubit 4

In [25]:
xmon_options = dict(
    pos_x = '4.0mm',
    pos_y = '2.15mm',
    orientation = '90',
    cross_width = '24um',
    cross_length = '160um',
    cross_gap = '24um',
    connection_pads=dict(
        connector_1 = dict(connector_location = '90', connector_type = '0', claw_length='90um')
    ),
)

transmon_options_4 = results_Qubit_1.values[0][15]
transmon_options_4.update(
    pos_x = '4.0mm',
    pos_y = '2.15mm',
    orientation = '90',
    )

Qubit_4 = TransmonCross(design, 'Qubit_4', options=transmon_options_4)
gui.rebuild()
gui.autoscale()
gui.zoom_on_components(['Qubit_4'])

### Qubit 5

In [26]:
xmon_options = dict(
    pos_x = '1.0mm',
    pos_y = '1.45mm',
    orientation = '270',
    cross_width = '24um',
    cross_length = '160um',
    cross_gap = '24um',
    connection_pads=dict(
        connector_1 = dict(connector_location = '90', connector_type = '0', claw_length='90um')
    ),
)
transmon_options_5 = results_Qubit_1.values[0][15]
transmon_options_5.update(
    pos_x = '1.0mm',
    pos_y = '1.45mm',
    orientation = '270',)

Qubit_5 = TransmonCross(design, 'Qubit_5', options=transmon_options_5)
gui.rebuild()
gui.autoscale()
gui.zoom_on_components(['Qubit_5'])

### Qubit 6

In [27]:
xmon_options = dict(
    pos_x = '4.0mm',
    pos_y = '1.0mm',
    orientation = '90',
    cross_width = '24um',
    cross_length = '160um',
    cross_gap = '24um',
    connection_pads=dict(
        connector_1 = dict(connector_location = '90', connector_type = '0', claw_length='90um')
    ),
)

transmon_options_6 = results_Qubit_1.values[0][15]
transmon_options_6.update(
    pos_x = '4.0mm',
    pos_y = '1.00mm',
    orientation = '90',
    )

Qubit_6 = TransmonCross(design, 'Qubit_6', options=transmon_options_6)
gui.rebuild()
gui.autoscale()
gui.zoom_on_components(['Qubit_6'])

In [28]:
LaunchpadWirebondDriven.default_options

{'trace_width': 'cpw_width',
 'trace_gap': 'cpw_gap',
 'lead_length': '25um',
 'pad_width': '80um',
 'pad_height': '80um',
 'pad_gap': '58um',
 'taper_height': '122um'}

# Readout line

### Readout launchpad

In [29]:
output = LaunchpadWirebondDriven(
                                 design, 
                                 'output', 
                                 options = dict(
                                     pos_x='2500um', 
                                     pos_y='600um', 
                                     orientation='90', 
                                     lead_length='30um',
                                     pad_gap='100um',
                                     pad_width='160um',
                                     pad_height='200um',
                                     taper_height = '200um',
                                    )
                                )
input = LaunchpadWirebondDriven(
                                 design, 
                                 'input', 
                                 options = dict(
                                     pos_x='2500um', 
                                     pos_y='4400um', 
                                     orientation='270', 
                                     lead_length='30um',
                                     pad_gap='100um',
                                     pad_width='160um',
                                     pad_height='200um',
                                     taper_height = '200um',
                                    )
                                )
gui.rebuild()
gui.autoscale()

## Readout line

In [30]:
IObus = RouteStraight(design,'IObus',options=Dict(
    pin_inputs=Dict(
        start_pin=Dict(
            component = 'input',
            pin = 'tie'
        ),
        end_pin=Dict(
            component = 'output',
            pin = 'tie'
        )
    )
))
gui.rebuild()
# gui.autoscale()

# Readout Resonator helpers

In [31]:
# the unit is um in this box

pos_ro_x = 2500
cpw_width = 10
epsilon = 11.4
fillet='99.99um'
cpw_options = Dict(
    hfss_wire_bonds = True,
    lead=Dict(
        start_straight='100um',
        # end_straight='250um'
    ),
    fillet=fillet,
    meander=Dict(spacing="200um")
)


def pos_from_offset(offset):
    return pos_ro_x + offset

def quarter_wave_length(frequency_ghz, epsilon_eff):
    """input in GHz, output in um"""
    c = 3e8
    frequency_hz = frequency_ghz * 1e9
    wavelength = c / (frequency_hz * (epsilon_eff ** 0.5))
    quarter_wavelength = wavelength / 4
    return quarter_wavelength * 1e6

def generate_readout_frequencies(center_freq_ghz=7.4, spacing_mhz=100, num_qubits=6):
    start_freq = center_freq_ghz - (spacing_mhz * (num_qubits - 1) / 2) / 1000
    return [round(start_freq + i * spacing_mhz / 1000, 6) for i in range(num_qubits)]

def connect(cpw_name: str, pin1_comp_name: str, pin1_comp_pin: str, pin2_comp_name: str, pin2_comp_pin: str,
            length: str, asymmetry='0 um'):
    """Connect two pins with a CPW."""
    myoptions = Dict(
        total_length=length,
        pin_inputs=Dict(
            start_pin=Dict(component=pin1_comp_name,pin=pin1_comp_pin),
            end_pin=Dict(component=pin2_comp_name,pin=pin2_comp_pin),
            ),
        )
    myoptions.update(cpw_options)
    myoptions.meander.asymmetry = asymmetry
    return RouteMeander(design, cpw_name, myoptions)

frequencies = generate_readout_frequencies()

length_um_list = [quarter_wave_length(freq, epsilon) for freq in frequencies]

# length_um = quarter_wave_length(7.4, 6.3)
# for length_um in length_um_list:
#     print(f"{length_um:.2f}")


In [32]:
RouteMeander.get_template_options(design)

{'chip': 'main',
 'layer': '1',
 'pin_inputs': {'start_pin': {'component': '', 'pin': ''},
  'end_pin': {'component': '', 'pin': ''}},
 'fillet': '0',
 'lead': {'start_straight': '0mm',
  'end_straight': '0mm',
  'start_jogged_extension': '',
  'end_jogged_extension': ''},
 'total_length': '7mm',
 'trace_width': 'cpw_width',
 'meander': {'spacing': '200um', 'asymmetry': '0um'},
 'snap': 'true',
 'prevent_short_edges': 'true',
 'hfss_wire_bonds': False,
 'q3d_wire_bonds': False,
 'aedt_q3d_wire_bonds': False,
 'aedt_hfss_wire_bonds': False}

# Readout Resonators

In [33]:
offset_ro_x_1 = -28
pos_x_cp1 = pos_from_offset(offset_ro_x_1)
pos_y_cp1 = 4040

# coupling_pin_1 = OpenToGround(design, 'coupling_pin_1', options=dict(
#     pos_x = f"{pos_x_cp1}um",
#     pos_y = f"{pos_y_cp1}um",
#     orientation = "270.0"
# ))

coupling_pin_1 = ShortToGround(design, 'coupling_pin_1', options=dict(
    pos_x = f"{pos_x_cp1}um",
    pos_y = f"{pos_y_cp1}um",
    orientation = "270.0"
))

asym = 0
cpw1 = connect('cpw1', 'Qubit_1', 'readout', 'coupling_pin_1', 'short', '4037um', f'+{asym}um')

gui.rebuild()
# gui.autoscale()


In [34]:
offset_ro_x_2 = +28
pos_x_cp2 = pos_from_offset(offset_ro_x_2)
pos_y_cp2 = 3570

coupling_pin_2 = ShortToGround(design, 'coupling_pin_2', options=dict(
    pos_x = f"{pos_x_cp2}um",
    pos_y = f"{pos_y_cp2}um",
    orientation = "90.0"
))

asym = 0
cpw2 = connect('cpw2', 'Qubit_2', 'readout', 'coupling_pin_2', 'short', f"{length_um_list[1]}um", f'+{asym}um')

gui.rebuild()
gui.autoscale()

In [35]:
offset_ro_x_3 = -28
pos_x_cp3 = pos_from_offset(offset_ro_x_3)
pos_y_cp3 = 2840

coupling_pin_3 = ShortToGround(design, 'coupling_pin_3', options=dict(
    pos_x = f"{pos_x_cp3}um",
    pos_y = f"{pos_y_cp3}um",
    orientation = "270.0"
))

asym = 0
cpw3 = connect('cpw3', 'Qubit_3', 'readout', 'coupling_pin_3', 'short', f"{length_um_list[2]}um", f'+{asym}um')

gui.rebuild()
gui.autoscale()

In [36]:
offset_ro_x_4 = +28
pos_x_cp4 = pos_from_offset(offset_ro_x_4)
pos_y_cp4 = 2170

coupling_pin_4 = ShortToGround(design, 'coupling_pin_4', options=dict(
    pos_x = f"{pos_x_cp4}um",
    pos_y = f"{pos_y_cp4}um",
    orientation = "90.0"
))

asym = 0
cpw4 = connect('cpw4', 'Qubit_4', 'readout', 'coupling_pin_4', 'short', f"{length_um_list[3]}um", f'+{asym}um')

gui.rebuild()
gui.autoscale()

In [37]:
offset_ro_x_5 = -28
pos_x_cp5 = pos_from_offset(offset_ro_x_5)
pos_y_cp5 = 1430

coupling_pin_5 = ShortToGround(design, 'coupling_pin_5', options=dict(
    pos_x = f"{pos_x_cp5}um",
    pos_y = f"{pos_y_cp5}um",
    orientation = "270.0"
))

asym = 0
cpw5 = connect('cpw5', 'Qubit_5', 'readout', 'coupling_pin_5', 'short', f"{length_um_list[4]}um", f'+{asym}um')

gui.rebuild()
gui.autoscale()

In [38]:
offset_ro_x_6 = +28
pos_x_cp6 = pos_from_offset(offset_ro_x_6)
pos_y_cp6 = 1030

coupling_pin_6 = ShortToGround(design, 'coupling_pin_6', options=dict(
    pos_x = f"{pos_x_cp6}um",
    pos_y = f"{pos_y_cp6}um",
    orientation = "90.0"
))

asym = 0
cpw6 = connect('cpw6', 'Qubit_6', 'readout', 'coupling_pin_6', 'short', f"{length_um_list[5]}um", f'+{asym}um')

gui.rebuild()
gui.autoscale()

In [39]:
a_gds = design.renderers.gds
a_gds.options['path_filename'] = './qiskit-metal/tutorials/resources/Fake_Junctions.GDS'
a_gds.options.no_cheese
a_gds.options['no_cheese']['view_in_file']['main'][1] = False
a_gds.options['cheese']['view_in_file']['main'][1] = False
a_gds.options['short_segments_to_not_fillet'] = True
scale_fillet = 40
a_gds.options['check-short_segments_by_scaling_fillet'] = scale_fillet
a_gds.options['tolerance'] = '0.00001'
a_gds.export_to_gds('candle.gds')

03:56PM 11s WARNING [_import_junction_gds_file]: Not able to find file:"./qiskit-metal/tutorials/resources/Fake_Junctions.GDS".  Not used to replace junction. Checked directory:"c:\Users\91566\Desktop\rz\Qubit_Design\qubit_design\qiskit-metal\tutorials\resources".


1

# Analysis

In [40]:
from qiskit_metal.analyses.quantization import EPRanalysis
eig_qb = EPRanalysis(design, "hfss")

In [41]:
hfss = eig_qb.sim.renderer

In [42]:
hfss.start()

INFO 03:56PM [connect_project]: Connecting to Ansys Desktop API...
INFO 03:56PM [load_ansys_project]: 	Opened Ansys App
INFO 03:56PM [load_ansys_project]: 	Opened Ansys Desktop v2025.1.0
INFO 03:56PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/91566/Documents/Ansoft/
	Project:   Project65
INFO 03:56PM [connect_design]: No active design found (or error getting active design).
INFO 03:56PM [connect]: 	 Connected to project "Project65". No design detected


True

In [43]:
hfss.activate_ansys_design("Transmon_cpw", "eigenmode")

03:56PM 33s WARNING [activate_ansys_design]: The design_name=Transmon_cpw was not in active project.  Designs in active project are: 
[].  A new design will be added to the project.  
INFO 03:56PM [connect_design]: 	Opened active design
	Design:    Transmon_cpw [Solution type: Eigenmode]
WARNING 03:56PM [connect_setup]: 	No design setup detected.
WARNING 03:56PM [connect_setup]: 	Creating eigenmode default setup.
INFO 03:56PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)


In [44]:
# hfss.render_design(['Qubit_1','cpw1'],[])

In [45]:
eig_qb.sim.setup.name = 'Tune_Qubit_1'
eig_qb.sim.setup.freq_ghz = 4
eig_qb.sim.setup.min_freq_ghz = 3
eig_qb.sim.setup.max_passes = 12
eig_qb.sim.setup.max_delta_f = 0.1
eig_qb.sim.setup.n_modes = 3
eig_qb.sim.setup

{'name': 'Tune_Qubit_1',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'min_freq_ghz': 3,
 'n_modes': 3,
 'max_delta_f': 0.1,
 'max_passes': 12,
 'min_passes': 1,
 'min_converged': 1,
 'pct_refinement': 30,
 'basis_order': 1,
 'vars': {'Lj': '10 nH', 'Cj': '0 fF'},
 'freq_ghz': 4}

## below is something from the past

In [46]:
# Qubit_1.options.hfss_inductance = '15nH'

In [47]:
del eig_qb.setup.junctions['jj']
eig_qb.setup.junctions.jj = Dict(rect='JJ_rect_Lj_Qubit_1_rect_jj', line='JJ_Lj_Qubit_1_rect_jj_',
                                 Lj_variable = 'Lj', Cj_variable = 'Cj')
eig_qb.setup.sweep_variable = 'Lj'
eig_qb.setup

{'junctions': {'jj': {'rect': 'JJ_rect_Lj_Qubit_1_rect_jj',
   'line': 'JJ_Lj_Qubit_1_rect_jj_',
   'Lj_variable': 'Lj',
   'Cj_variable': 'Cj'}},
 'dissipatives': {'dielectrics_bulk': ['main']},
 'cos_trunc': 8,
 'fock_trunc': 7,
 'sweep_variable': 'Lj'}

In [48]:
eig_qb.sim.run(name="Qubit_1", components=['Qubit_1', 'cpw1'], open_terminations=[])

INFO 03:56PM [connect_design]: 	Opened active design
	Design:    Qubit_1_hfss [Solution type: Eigenmode]
WARNING 03:56PM [connect_setup]: 	No design setup detected.
WARNING 03:56PM [connect_setup]: 	Creating eigenmode default setup.
INFO 03:56PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 03:56PM [get_setup]: 	Opened setup `Tune_Qubit_1`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 03:56PM [analyze]: Analyzing setup Tune_Qubit_1
03:58PM 26s INFO [get_f_convergence]: Saved convergences to c:\Users\91566\Desktop\rz\Qubit_Design\qubit_design\hfss_eig_f_convergence.csv


In [49]:
eig_qb.sim.plot_convergences()

In [50]:
eig_qb.run_epr()

Design "Qubit_1_hfss" info:
	# eigenmodes    3
	# variations    1
Design "Qubit_1_hfss" info:
	# eigenmodes    3
	# variations    1


 c:\Users\91566\miniconda3\envs\sq\Lib\site-packages\pyEPR\project_info.py: 239



        energy_elec_all       = 1.80610547588938e-24
        energy_elec_substrate = 1.64720548521564e-24
        EPR of substrate = 91.2%

        energy_mag    = 1.52915729394836e-26
        energy_mag % of energy_elec_all  = 0.8%
        

Variation 0  [1/1]


 c:\Users\91566\miniconda3\envs\sq\Lib\site-packages\pyEPR\core_distributed_analysis.py: 1101
 c:\Users\91566\miniconda3\envs\sq\Lib\site-packages\pyEPR\core_distributed_analysis.py: 1102
 c:\Users\91566\miniconda3\envs\sq\Lib\site-packages\pyEPR\core_distributed_analysis.py: 1245



  Mode 0 at 4.95 GHz   [1/3]
    Calculating ℰ_magnetic,

...

###### Numerical diagonalization

Lj,10
0,4760.60
1,6782.66
2,20330.51


#### Kerr Non-linear coefficient table (MHz)

###### Numerical diagonalization

0         1         2
Lj                              
10 0  200.55  1.81e-01  1.76e-02
   1    0.18  4.63e-05  8.55e-06
   2    0.02  8.55e-06  4.44e-07

In [51]:
eig_qb.sim.close()